# Generation speed in one Kaggle T4 session

Attach `spritker/vitok-code`, `spritker/vitok-data`, and the outputs of both
`spritker/nanovitok-03-full-a` and `spritker/nanovitok-03-full-b`.
This notebook performs no training. It measures the original model, both zero-shot
extensions, and all four final models on the same first 20 test prompts, with
256 greedy tokens and special tokens banned, as fixed in PLAN.md.

The small `vi` scores generated here only verify that the speed inputs pair;
use notebook 03's full 2,000-document scores for quality conclusions.
Each output records one session ID. Timings from an older session cannot be reused.
Two queues keep both T4s busy; the session has a 60-minute deadline, including setup.
Measured prefill plus decode work is 1,031.666 seconds (17.194 minutes) for
the longer queue and 777.055 seconds (12.951 minutes) for the other queue, summed
from notebook 03 generation records. Setup, model loading, the 20-document
verification scores, and input mounting time are unmeasured. The deadline is a cap, not a completion-time estimate.

In [ ]:
import os, time
from pathlib import Path
START = time.monotonic()
INPUT = Path(os.environ.get("VITOK_KAGGLE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("VITOK_KAGGLE_WORK", "/kaggle/working/generation-speed"))
MODEL, REVISION = "Qwen/Qwen3-0.6B", "c1899de289a04d12100db370d81485cdf75e47ca"
DTYPE, SPEED_PROMPTS, PROMPT_CHARS, NEW_TOKENS = "fp16", 20, 500, 256
SESSION_MINUTES = 60
QUEUES = {0: ["original", "base", "syllable-1000-before", "syllable-1000"],
          1: ["multisyllable-1000-before", "multisyllable-1000", "multisyllable-1000-s1"]}

In [ ]:
import hashlib, importlib.util, json, shutil, subprocess, sys, uuid
from importlib.metadata import version
if (WORK / "session-manifest.json").exists():
    raise RuntimeError("use a new empty work directory: timing records may not span sessions")
bundle = next(p.parent for p in INPUT.rglob("pyproject.toml") if (p.parent / "src" / "vitok").is_dir())
if importlib.util.find_spec("vitok") is None:
    code = Path("/tmp/vitok-speed")
    shutil.copytree(bundle, code, dirs_exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(code)], check=True)
    sys.path.insert(0, str(code / "src"))
from vitok import kaggle_session as ks
inputs = ks.find_inputs(INPUT)
R, D = inputs["code"] / "retrofit", inputs["data"]
ks.check_versions({p: version(p) for p in ("torch", "transformers", "tokenizers")},
                  {"torch": "2.4", "transformers": "4.56", "tokenizers": "0.21"})
gpus = ks.preflight(need_gpus=2)
base_model = ks.prefetch(MODEL, REVISION)
WORK.mkdir(parents=True, exist_ok=True)
lines = (D / "test.jsonl").read_text(encoding="utf-8").splitlines()[:SPEED_PROMPTS]
if len(lines) != SPEED_PROMPTS:
    raise RuntimeError("not enough test documents for the fixed speed prompts")
prompt_file = WORK / "speed_docs.jsonl"
prompt_file.write_text("\n".join(lines) + "\n", encoding="utf-8")
session_id = str(uuid.uuid4())
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
jobs = {"original": {"model": base_model, "tokenizer": str(R / "base" / "tokenizer.json"), "snapshot": None}}
for name in ("base", "syllable-1000", "multisyllable-1000", "multisyllable-1000-s1"):
    candidates = [p.parent.parent for p in INPUT.rglob("final.pt")
                  if p.parent.name == "train" and p.parent.parent.name == name]
    if len(candidates) != 1:
        raise RuntimeError(f"expected one attached training output for {name}, got {candidates}")
    root = candidates[0]
    weights = root / "train" / "final.pt"
    expected = json.loads((root / "score-final" / "summary.json").read_text())["signature"]["snapshot_sha256"]
    if sha(weights) != expected:
        raise RuntimeError(f"checkpoint differs from the scored final for {name}")
    manifest = json.loads((root / "train" / "manifest.json").read_text())
    if tuple(manifest["signature"][k] for k in ("steps", "batch", "seq_len")) != (1200, 32, 512):
        raise RuntimeError(f"unexpected training budget for {name}")
    kind = name.removesuffix("-s1")
    model = base_model if name == "base" else str(root / "extended")
    if name != "base":
        ext = json.loads((root / "extended" / "extend_manifest.json").read_text())
        if (ext["model"], ext["revision"]) != (MODEL, REVISION):
            raise RuntimeError(f"wrong starting-model revision for {name}")
    jobs[name] = {"model": model, "tokenizer": str(R / kind / "tokenizer.json"), "snapshot": str(weights),
                  "snapshot_sha256": expected, "training_commit": manifest["git_commit"]}
    if name in ("syllable-1000", "multisyllable-1000"):
        jobs[name + "-before"] = {"model": model, "tokenizer": jobs[name]["tokenizer"], "snapshot": None}
for job in jobs.values():
    job["tokenizer_sha256"] = sha(Path(job["tokenizer"]))
cfg = {"session_id": session_id, "jobs": jobs, "queues": QUEUES, "work": str(WORK),
       "prompt_file": str(prompt_file), "prompt_sha256": sha(prompt_file), "dtype": DTYPE,
       "speed_prompts": SPEED_PROMPTS, "prompt_chars": PROMPT_CHARS, "new_tokens": NEW_TOKENS,
       "deadline": START + SESSION_MINUTES * 60, "gpus": gpus,
       "packages": {p: version(p) for p in ("torch", "transformers", "tokenizers")}}
config_path = WORK / "session-manifest.json"
config_path.write_text(json.dumps(cfg, indent=2))
print(json.dumps({"session_id": session_id, "models": list(jobs), "gpus": gpus}, indent=2))

In [ ]:
worker = r"""
import hashlib, json, os, subprocess, sys, time
from pathlib import Path
config = json.loads(Path(sys.argv[1]).read_text())
gpu = sys.argv[2]
work = Path(config["work"])
outcomes = {}
for name in config["queues"][gpu]:
    job = config["jobs"][name]
    out = work / name
    argv = [sys.executable, "-m", "vitok.score", "--model", job["model"], "--tokenizer", job["tokenizer"],
            "--vi-docs", config["prompt_file"], "--speed-prompts", str(config["speed_prompts"]),
            "--speed-prompt-chars", str(config["prompt_chars"]), "--new-tokens", str(config["new_tokens"]),
            "--dtype", config["dtype"], "--out", str(out)]
    if job["snapshot"]:
        argv += ["--snapshot", job["snapshot"]]
    left = config["deadline"] - time.monotonic()
    if left <= 0:
        outcomes[name] = {"status": "incomplete", "reason": "session deadline"}
    else:
        argv += ["--deadline-minutes", str(left / 60)]
        try:
            with open(work / (name + ".log"), "a") as log:
                log.write("$ " + " ".join(argv) + "\n")
                log.flush()
                code = subprocess.run(argv, stdout=log, stderr=subprocess.STDOUT, timeout=left).returncode
            outcomes[name] = {"status": "done" if code == 0 else "incomplete" if code == 3 else "failed",
                              "exit_code": code}
        except subprocess.TimeoutExpired:
            outcomes[name] = {"status": "incomplete", "reason": "session deadline during model loading/scoring"}
        except Exception as e:
            outcomes[name] = {"status": "failed", "reason": f"{type(e).__name__}: {e}"}
    outcomes[name]["session_id"] = config["session_id"]
    outcomes[name]["gpu"] = int(gpu)
    (work / ("queue-gpu" + gpu + ".json")).write_text(json.dumps(outcomes, indent=2))
sys.exit(0 if all(r["status"] == "done" for r in outcomes.values()) else 1)
"""
worker_path = WORK / "speed_worker.py"
worker_path.write_text(worker)
processes, handles = {}, []
for gpu in QUEUES:
    log = open(WORK / f"worker-gpu{gpu}.log", "a")
    handles.append(log)
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "HF_HUB_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false"}
    processes[gpu] = subprocess.Popen([sys.executable, str(worker_path), str(config_path), str(gpu)],
                                      stdout=log, stderr=subprocess.STDOUT, env=env)
codes = {gpu: process.wait() for gpu, process in processes.items()}
for handle in handles:
    handle.close()
print("worker exit codes:", codes)

In [ ]:
results = {}
for gpu in QUEUES:
    path = WORK / f"queue-gpu{gpu}.json"
    if not path.exists():
        raise RuntimeError(f"worker {gpu} did not write outcomes; read its log")
    results.update(json.loads(path.read_text()))
for gpu, names in QUEUES.items():
    for name in names:
        if name not in results:
            results[name] = {"status": "failed", "reason": "worker stopped before reporting this model",
                             "session_id": session_id, "gpu": gpu}
for name, rec in results.items():
    if rec["status"] != "done":
        continue
    summary = json.loads((WORK / name / "summary.json").read_text())
    sig = summary["signature"]
    expected = jobs[name]
    if (sig["vi_docs_sha256"], sig["tokenizer_sha256"], sig["snapshot_sha256"],
        sig["speed_prompts"], sig["speed_prompt_chars"], sig["new_tokens"]) != (
        cfg["prompt_sha256"], expected["tokenizer_sha256"], expected.get("snapshot_sha256"),
        SPEED_PROMPTS, PROMPT_CHARS, NEW_TOKENS):
        raise RuntimeError(f"score signature does not match the speed session for {name}")
    if summary["generation"]["prompts"] != SPEED_PROMPTS:
        raise RuntimeError(f"missing speed prompts for {name}")
    rec["generation"] = summary["generation"]
    rec["device_name"] = summary["device_name"]
result = {"session_id": session_id, "status": "done" if set(results) == set(jobs) and all(code == 0 for code in codes.values())
          and all(r["status"] == "done" for r in results.values()) else "incomplete",
          "models": results, "wall_seconds": time.monotonic() - START}
(WORK / "session-results.json").write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
if result["status"] != "done":
    for path in sorted(WORK.glob("*.log")):
        print(path.name, path.read_text())
    raise RuntimeError("speed session incomplete; no cross-session timing reuse")